[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/01_GEE_Data_Preprocessing.ipynb)

# Notebook 01 - GEE Data Preprocessing
**Project:** WASHLAB Climate-Smart WASH Pilot - Kitui County  
**Analyst:** Davis Mironga  
**Output:** Preprocessed rasters exported as **GEE Assets** to `projects/kitui-washlab-analysis/assets/kitui/`

---

Pulls and exports all satellite datasets needed for the Water Access Stress Index (WASI) analysis in Notebook 02.

| Dataset | GEE collection | Resolution | Used in |
|---------|---------------|------------|---------|
| NDVI | MODIS MOD13A3 | 1 km monthly | WASI C3 - vegetation condition |
| NDVI baseline | MODIS MOD13A3 2000–2004 | 1 km | WASI C3 - anomaly reference |
| Rainfall baseline | CHIRPS 1981–2010 | 5 km | WASI C1 - long-term normal |
| Rainfall seasonal | CHIRPS long/short rains | 5 km | WASI C1 - seasonal deficit |
| Rainfall recent | CHIRPS 2020–2024 | 5 km | WASI C1 - current deficit |
| Elevation | SRTM 30 m | 30 m | WASI C4 - terrain accessibility |
| Slope | SRTM derived | 30 m | WASI C4 - terrain accessibility |
| Population | WorldPop 2020 | 100 m | WASI C2 - demand weighting |
| Surface water seasonality | JRC GSW | 30 m | WASI C5 - water reliability |
| Surface water occurrence | JRC GSW | 30 m | WASI C5 - water reliability |
| Surface water transition | JRC GSW | 30 m | WASI C5 - change detection |
| Land surface temperature | MODIS MOD11A2 | 1 km | Supporting layer |
| Evapotranspiration | MODIS MOD16A2 | 500 m | Supporting layer |
| Soil moisture | ERA5-Land monthly | ~9 km | Groundwater recharge proxy |

> ⚠️ **Run cells in order.** Do not start Notebook 02 until all exports show `COMPLETED`.
> Assets are stored in GEE (not Google Drive) — no Drive storage is consumed.


### 1. Setup

Installs dependencies and authenticates GEE. **No Drive mount needed** — all outputs go to GEE Assets.

⚠️ `GEE_PROJECT` is already set to `kitui-washlab-analysis`. Only change it if you use a different project ID.


In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install earthengine-api geemap geopandas -q

import ee
import geemap
import os

GEE_PROJECT  = 'kitui-washlab-analysis'
ASSET_FOLDER = f'projects/{GEE_PROJECT}/assets/kitui'

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

# Create the asset folder — silently skip if it already exists
try:
    ee.data.createAsset({'type': 'Folder'}, ASSET_FOLDER)
    print(f'Asset folder created: {ASSET_FOLDER}')
except Exception:
    print(f'Asset folder already exists: {ASSET_FOLDER}')

print('Setup complete')
print(f'Exports will go to GEE Assets: {ASSET_FOLDER}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.0 MB/s eta 0:00:00
Asset folder already exists: projects/kitui-washlab-analysis/assets/kitui
Setup complete
Exports will go to GEE Assets: projects/kitui-washlab-analysis/assets/kitui


### 2. Study Area

Loads Kitui County boundary from FAO GAUL (level 2) and displays it on the map.

All datasets pulled after this step are clipped to `kitui_geom`.

In [2]:
# ── 1. Define Kitui County study area ─────────────────────────────────────────
kitui = (ee.FeatureCollection('FAO/GAUL/2015/level2')
           .filter(ee.Filter.And(
               ee.Filter.eq('ADM0_NAME', 'Kenya'),
               ee.Filter.eq('ADM2_NAME', 'Kitui')
           )))

n = kitui.size().getInfo()
print(f'Features matched: {n}')

kitui_geom = kitui.geometry()

Map = geemap.Map()
Map.centerObject(kitui_geom, 8)
Map.addLayer(kitui_geom, {'color': '0B5394', 'fillColor': '0B539433'}, 'Kitui County')
print('Kitui County boundary loaded')
Map


Features matched: 1
Kitui County boundary loaded


Map(center=[-1.869528123546361, 38.446305795184365], controls=(WidgetControl(options=['position', 'transparent…

### 2b. Correct Study Area Extent

The FAO GAUL polygon for Kitui only extends to latitude -1.055 and does not cover the northern wards.
This cell overrides `kitui_geom` with a bounding box that covers all 40 wards.
This must be run before any data loading or export cell.


In [ ]:
# Override kitui_geom with full county bounding box
# The FAO GAUL polygon only covers to latitude -1.055 and misses the northern wards.
# This bbox covers all 40 wards as confirmed by the ward boundary shapefile.
kitui_geom = ee.Geometry.BBox(37.5, -3.1, 39.2, 0.0)
print('kitui_geom set to full county bbox: 37.5, -3.1, 39.2, 0.0')
print(kitui_geom.bounds().getInfo())


### 3. NDVI

Pulls MODIS monthly NDVI (MOD13A3) and computes two products:
- `ndvi_mean_2000_2025` - long-term mean vegetation condition
- `ndvi_baseline_2000_2004` - earliest available period used as anomaly reference in WASI C3

Note: MODIS starts in 2000 so 2000–2004 is used as the baseline rather than 1995–2005.

In [3]:
# ── 2. NDVI — MODIS MOD13A3 ───────────────────────────────────────────────────
# Two products exported:
#   a) Long-term mean 2000–2025 — used as current vegetation condition
#   b) Baseline mean 1995–2005  — used as the anomaly reference in WASI C3
#      (MODIS only starts 2000 — 2000–2004 used as earliest available baseline)

ndvi_coll = (ee.ImageCollection('MODIS/061/MOD13A3')
               .filterBounds(kitui_geom)
               .select('NDVI')
               .map(lambda img: img.multiply(0.0001)           # MODIS scale factor
                                   .copyProperties(img, ['system:time_start'])))

# a) 2000–2025 mean — current vegetation condition
ndvi_mean_2000_2025 = (ndvi_coll
                         .filterDate('2000-01-01', '2025-12-31')
                         .mean()
                         .clip(kitui_geom)
                         .rename('ndvi_mean'))

# b) 2000–2004 baseline — earliest 5-year window MODIS can provide
#    Label clearly in all outputs: MODIS baseline not 1995
ndvi_baseline_2000_2004 = (ndvi_coll
                             .filterDate('2000-01-01', '2004-12-31')
                             .mean()
                             .clip(kitui_geom)
                             .rename('ndvi_baseline'))

# Annual mean series for trend analysis (optional)
years = ee.List.sequence(2000, 2025)
def annual_ndvi(year):
    y = ee.Number(year).int()
    return (ndvi_coll
              .filter(ee.Filter.calendarRange(y, y, 'year'))
              .mean()
              .set('year', y)
              .set('system:time_start', ee.Date.fromYMD(y, 1, 1).millis()))

ndvi_annual = ee.ImageCollection(years.map(annual_ndvi))
print('NDVI collections ready')
print('  2000–2025 mean: current vegetation condition')
print('  2000–2004 baseline: anomaly reference (earliest MODIS window)')

NDVI collections ready
  2000–2025 mean: current vegetation condition
  2000–2004 baseline: anomaly reference (earliest MODIS window)


### 4. Rainfall

Pulls CHIRPS pentadal precipitation and computes four products:
- `rainfall_baseline_30yr` - 30-year WMO baseline 1981–2010 (denominator for deficit)
- `rainfall_short_rains` - October to December mean
- `rainfall_long_rains` - March to May mean
- `rainfall_recent_2020_2024` - recent annual total (numerator for deficit)

Deficit = recent total ÷ 30-year baseline. Values below 1.0 indicate drier-than-normal conditions.

In [4]:
# ── 3. Rainfall — CHIRPS v2.0 ─────────────────────────────────────────────────
# CHIRPS pentadal (5-day) precipitation, 1981–2025
# Exported products:
#   - 30-year baseline annual mean (1981–2010) — denominator for deficit
#   - Short rains mean (Oct–Dec)
#   - Long rains mean (Mar–May)
#   - Recent annual total (2020–2024) — numerator for deficit

chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
            .filterBounds(kitui_geom)
            .select('precipitation'))

# 30-year baseline annual mean (1981–2010 WMO standard period)
# Sum all pentads per year then average over 30 years
baseline_years = ee.List.sequence(1981, 2010)
def annual_rain(year):
    y = ee.Number(year).int()
    return (chirps.filter(ee.Filter.calendarRange(y, y, 'year'))
                  .sum()
                  .set('year', y)
                  .set('system:time_start', ee.Date.fromYMD(y, 1, 1).millis()))

rainfall_baseline_30yr = (ee.ImageCollection(baseline_years.map(annual_rain))
                            .mean()
                            .clip(kitui_geom)
                            .rename('rainfall_baseline_mm_yr'))

# Recent 5-year annual total mean (2020–2024) — for deficit calculation
recent_years = ee.List.sequence(2020, 2024)
rainfall_recent_5yr = (ee.ImageCollection(recent_years.map(annual_rain))
                         .mean()
                         .clip(kitui_geom)
                         .rename('rainfall_recent_mm_yr'))

# Seasonal composites
short_rains = (chirps.filter(ee.Filter.calendarRange(10, 12, 'month'))
                     .filterDate('2000-01-01', '2025-12-31')
                     .mean()
                     .clip(kitui_geom)
                     .rename('short_rains_mean_pentad'))

long_rains  = (chirps.filter(ee.Filter.calendarRange(3, 5, 'month'))
                     .filterDate('2000-01-01', '2025-12-31')
                     .mean()
                     .clip(kitui_geom)
                     .rename('long_rains_mean_pentad'))

print('Rainfall layers ready')
print('  Baseline: 1981–2010 (WMO 30-year standard)')
print('  Recent:   2020–2024 (5-year mean)')
print('  Seasonal: short rains (Oct–Dec), long rains (Mar–May)')

Rainfall layers ready
  Baseline: 1981–2010 (WMO 30-year standard)
  Recent:   2020–2024 (5-year mean)
  Seasonal: short rains (Oct–Dec), long rains (Mar–May)


### 5. Terrain

Loads SRTM elevation at 30 m and derives slope and aspect.

- `elev` - elevation in metres
- `slope` - slope in degrees, used in WASI C4 (accessibility)
- `aspect` - direction the land faces, for field context only, not used in WASI

In [5]:
# ── 4. Terrain — SRTM 30m ─────────────────────────────────────────────────────
srtm  = ee.Image('USGS/SRTMGL1_003').clip(kitui_geom)
slope = ee.Terrain.slope(srtm).rename('slope_deg')
elev  = srtm.rename('elevation_m')

# Aspect (for field context — not used in WASI)
aspect = ee.Terrain.aspect(srtm).rename('aspect_deg')

print('Terrain ready')
print('  Elevation, slope, aspect from SRTM 30m (USGS/SRTMGL1_003)')

Terrain ready
  Elevation, slope, aspect from SRTM 30m (USGS/SRTMGL1_003)


### 6. Population

Loads WorldPop Kenya 2020 at 100 m resolution, clipped to Kitui.

Used in WASI C2 to weight water stress by how many people are affected at each location.

In [6]:
# ── 5. Population density — WorldPop 2020 ─────────────────────────────────────
pop = (ee.ImageCollection('WorldPop/GP/100m/pop')
         .filter(ee.Filter.eq('country', 'KEN'))
         .filter(ee.Filter.eq('year', 2020))
         .first()
         .clip(kitui_geom)
         .rename('population_100m'))

print('Population density ready')
print('  WorldPop Kenya 2020, 100m resolution')

Population density ready
  WorldPop Kenya 2020, 100m resolution


### 7. Surface Water

Loads JRC Global Surface Water (GSW1_4) and extracts three bands:
- `water_seasonality` - months per year water is present (0 = never, 12 = permanent)
- `water_occurrence_pct` - % of observations with water detected 1984–2021
- `water_transition` - whether water extent has increased, decreased, or stayed stable

These are the core inputs to WASI C5.

In [7]:
# ── 6. JRC Global Surface Water ───────────────────────────────────────────────
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(kitui_geom)

# Seasonality: 0=no water, 1-11=seasonal, 12=permanent
water_seasonality = jrc.select('seasonality').rename('water_seasonality')
# Occurrence: % of Landsat observations when water was present (1984–2021)
water_occurrence  = jrc.select('occurrence').rename('water_occurrence_pct')
# Transition: change between epochs (stable, new, lost, seasonal, etc.)
water_transition  = jrc.select('transition').rename('water_transition')

print('JRC Surface Water ready')
print('  Seasonality, occurrence, and transition from JRC/GSW1_4')

JRC Surface Water ready
  Seasonality, occurrence, and transition from JRC/GSW1_4


### 8. Land Surface Temperature

Loads MODIS MOD11A2 daytime LST, converts from Kelvin to Celsius, and computes:
- `lst_mean_celsius` - annual mean 2000–2025
- `lst_dry_mean` - dry season mean January to March

Used as a supporting stress layer. Not a primary WASI component.

In [8]:
# ── 7. Land Surface Temperature — MODIS MOD11A2 ───────────────────────────────
lst = (ee.ImageCollection('MODIS/061/MOD11A2')
         .filterDate('2000-01-01', '2025-12-31')
         .filterBounds(kitui_geom)
         .select('LST_Day_1km')
         .map(lambda img: img.multiply(0.02)
                             .subtract(273.15)   # Kelvin → Celsius
                             .copyProperties(img, ['system:time_start'])))

lst_mean = lst.mean().clip(kitui_geom).rename('lst_mean_celsius')

# Seasonal LST for dry season heat stress
lst_dry = (lst.filter(ee.Filter.calendarRange(1, 3, 'month'))   # Jan–Mar dry season
              .mean().clip(kitui_geom).rename('lst_dry_season_celsius'))

print('Land Surface Temperature ready')
print('  Annual mean and dry-season (Jan–Mar) mean from MODIS MOD11A2')

Land Surface Temperature ready
  Annual mean and dry-season (Jan–Mar) mean from MODIS MOD11A2


### 9. Evapotranspiration

Loads MODIS MOD16A2 8-day ET, applies scale factor (× 0.1), and computes long-term mean.

`et_mean_mm_8day` - average water loss from land surface and vegetation. Used as a supporting layer alongside rainfall deficit.

In [9]:
# ── 8. Evapotranspiration — MODIS MOD16A2 ────────────────────────────────────
et = (ee.ImageCollection('MODIS/061/MOD16A2')
        .filterDate('2000-01-01', '2025-12-31')
        .filterBounds(kitui_geom)
        .select('ET')
        .map(lambda img: img.multiply(0.1)   # scale factor: kg/m²/8day → mm/8day
                            .copyProperties(img, ['system:time_start'])))

et_mean = et.mean().clip(kitui_geom).rename('et_mean_mm_8day')

print('Evapotranspiration ready')
print('  Annual mean 8-day ET from MODIS MOD16A2 (mm/8-day)')

Evapotranspiration ready
  Annual mean 8-day ET from MODIS MOD16A2 (mm/8-day)


### 10. Soil Moisture

Loads ERA5-Land monthly volumetric soil water (layer 1, 0–7 cm depth) and computes mean and seasonal patterns.

Used as a groundwater recharge proxy and for seasonal water availability context. Not a direct WASI input.

In [10]:
# ── 9. Soil moisture — ERA5-Land ──────────────────────────────────────────────
# ERA5-Land volumetric soil water in layer 1 (0–7 cm depth)
# Used as a groundwater recharge proxy — not a WASI component but useful
# for field validation and seasonal water availability narrative

era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
          .filterDate('2000-01-01', '2024-12-31')
          .filterBounds(kitui_geom)
          .select('volumetric_soil_water_layer_1'))

soil_moisture_mean = (era5.mean()
                         .clip(kitui_geom)
                         .rename('soil_moisture_mean_m3m3'))

# Seasonal: dry season (Jan–Mar) vs wet season (Mar–May)
soil_dry = (era5.filter(ee.Filter.calendarRange(1, 3, 'month'))
                .mean().clip(kitui_geom).rename('soil_moisture_dry_m3m3'))

soil_wet = (era5.filter(ee.Filter.calendarRange(3, 5, 'month'))
                .mean().clip(kitui_geom).rename('soil_moisture_wet_m3m3'))

print('Soil moisture ready')
print('  ERA5-Land volumetric soil water layer 1 (0–7cm), ~9km resolution')
print('  Annual mean, dry-season mean, wet-season mean')

Soil moisture ready
  ERA5-Land volumetric soil water layer 1 (0–7cm), ~9km resolution
  Annual mean, dry-season mean, wet-season mean


### 11. Export to GEE Assets

Exports all 19 layers as GEE Assets to `projects/kitui-washlab-analysis/assets/kitui/`.

**Why Assets instead of Drive?**  
- Your Drive is at 94% capacity — 19 GeoTIFFs would fill it  
- GEE Assets have a separate 250 GB quota  
- Notebook 02 reads assets directly via the `ee` API — no download needed  
- Assets are tied to the project, not your personal Drive

Most layers exported at 500 m. WorldPop at 100 m. SRTM slope/elevation at 30 m.

⚠️ Exports run as background tasks. Move to the next cell to monitor progress.


In [11]:
# ── 10. Export all layers to GEE Assets ──────────────────────────────────────
# Exports to projects/kitui-washlab-analysis/assets/kitui/
# Skips any asset that already exists — re-run is safe and will not overwrite completed exports.

BASE_PARAMS = {
    'region':    kitui_geom,
    'crs':       'EPSG:4326',
    'maxPixels': 1e13,
}

# (image, asset_name, scale_m)
EXPORTS = [
    # NDVI
    (ndvi_mean_2000_2025,       'kitui_ndvi_mean_2000_2025',         500),
    (ndvi_baseline_2000_2004,   'kitui_ndvi_baseline_2000_2004',     500),
    # Rainfall
    (rainfall_baseline_30yr,    'kitui_rainfall_baseline_1981_2010', 5000),
    (rainfall_recent_5yr,       'kitui_rainfall_recent_2020_2024',   5000),
    (short_rains,               'kitui_short_rains_mean',            5000),
    (long_rains,                'kitui_long_rains_mean',             5000),
    # Terrain
    (elev,                      'kitui_elevation_srtm30',              30),
    (slope,                     'kitui_slope_deg',                     30),
    (aspect,                    'kitui_aspect_deg',                    30),
    # Population
    (pop,                       'kitui_worldpop_2020',                100),
    # Surface water
    (water_seasonality,         'kitui_jrc_water_seasonality',         30),
    (water_occurrence,          'kitui_jrc_water_occurrence',          30),
    (water_transition,          'kitui_jrc_water_transition',          30),
    # Temperature
    (lst_mean,                  'kitui_lst_mean_celsius',            1000),
    (lst_dry,                   'kitui_lst_dry_season_celsius',      1000),
    # Evapotranspiration
    (et_mean,                   'kitui_et_mean_mm_8day',              500),
    # Soil moisture
    (soil_moisture_mean,        'kitui_soil_moisture_mean',          9000),
    (soil_dry,                  'kitui_soil_moisture_dry',           9000),
    (soil_wet,                  'kitui_soil_moisture_wet',           9000),
]

tasks    = []
skipped  = []
for image, asset_name, scale in EXPORTS:
    asset_id = f'{ASSET_FOLDER}/{asset_name}'

    # Skip if asset already exists
    try:
        ee.data.getAsset(asset_id)
        skipped.append(asset_name)
        print(f'  Skipped (exists): {asset_name}')
        continue
    except Exception:
        pass  # asset does not exist — proceed to export

    task = ee.batch.Export.image.toAsset(
        image=image,
        description=asset_name,
        assetId=asset_id,
        scale=scale,
        **BASE_PARAMS
    )
    task.start()
    tasks.append((asset_name, task))
    print(f'  Started: {asset_name}  ({scale}m)')

print()
print(f'{len(tasks)} export tasks submitted, {len(skipped)} already existed and were skipped.')
if tasks:
    print('Check progress at: https://code.earthengine.google.com/tasks')
    print('Exports take 5-30 minutes each depending on resolution and area.')
if len(skipped) == len(EXPORTS):
    print('All assets already exist. Proceed to the monitor cell to verify, then Notebook 02.')


  Skipped (exists): kitui_ndvi_mean_2000_2025
  Skipped (exists): kitui_ndvi_baseline_2000_2004
  Skipped (exists): kitui_rainfall_baseline_1981_2010
  Skipped (exists): kitui_rainfall_recent_2020_2024
  Skipped (exists): kitui_short_rains_mean
  Skipped (exists): kitui_long_rains_mean
  Skipped (exists): kitui_elevation_srtm30
  Skipped (exists): kitui_slope_deg
  Skipped (exists): kitui_aspect_deg
  Skipped (exists): kitui_worldpop_2020
  Skipped (exists): kitui_jrc_water_seasonality
  Skipped (exists): kitui_jrc_water_occurrence
  Skipped (exists): kitui_jrc_water_transition
  Skipped (exists): kitui_lst_mean_celsius
  Skipped (exists): kitui_lst_dry_season_celsius
  Skipped (exists): kitui_et_mean_mm_8day
  Skipped (exists): kitui_soil_moisture_mean
  Skipped (exists): kitui_soil_moisture_dry
  Skipped (exists): kitui_soil_moisture_wet

0 export tasks submitted, 19 already existed and were skipped.
All assets already exist. Proceed to the monitor cell to verify, then Notebook 02.


### 12. Monitor Exports

Re-run this cell periodically until all tasks show `COMPLETED`.

Also monitor at: [code.earthengine.google.com/tasks](https://code.earthengine.google.com/tasks)

⚠️ Do not open Notebook 02 until all 19 tasks are complete. Once complete, verify assets appear at:  
`projects/kitui-washlab-analysis/assets/kitui/`


In [12]:
# ── 11. Monitor export tasks ──────────────────────────────────────────────────
# Checks GEE task status if exports were submitted this session.
# If all assets already existed and were skipped, reports that directly.

if not tasks:
    print('No tasks submitted this session — all assets already existed.')
    print('Run the Asset Verification cell below to confirm all 19 are present.')
else:
    statuses = {}
    for filename, task in tasks:
        status = task.status()['state']
        statuses[filename] = status

    completed = sum(1 for s in statuses.values() if s == 'COMPLETED')
    running   = sum(1 for s in statuses.values() if s == 'RUNNING')
    pending   = sum(1 for s in statuses.values() if s == 'READY')
    failed    = sum(1 for s in statuses.values() if s == 'FAILED')

    print(f'Task status: {completed} done | {running} running | {pending} pending | {failed} failed')
    print()
    for filename, status in statuses.items():
        icon = '✅' if status == 'COMPLETED' else '🔄' if status == 'RUNNING' else '⏳' if status == 'READY' else '❌'
        print(f'  {icon} {filename}: {status}')

    if failed > 0:
        print('\nFailed tasks detected. Re-run the export cell — it will only re-submit the missing assets.')
    if completed == len(tasks):
        print('\nAll submitted exports complete. Run the Asset Verification cell below to confirm.')


No tasks submitted this session — all assets already existed.
Run the Asset Verification cell below to confirm all 19 are present.


### 13. Asset Verification + QA Map

Confirms all 19 assets exist in GEE and visualises key layers. Run after all exports complete.

**Confirm:**
- All 19 assets listed as `✅ EXISTS`
- Kitui boundary loads correctly in blue
- NDVI shows green in higher-rainfall areas, brown in drier areas
- Rainfall gradient is visible across the county
- Population concentrates around Kitui town and main roads

If any asset shows `❌ MISSING`, re-run the export cell and re-submit that specific layer.


In [13]:
# ── 12. Asset verification + QA visualisation ────────────────────────────────
# Step 1: Confirm all 19 assets exist in GEE
# Step 2: Visualise only the assets that exist (safe to run at any time)

EXPECTED_ASSETS = [
    'kitui_ndvi_mean_2000_2025',
    'kitui_ndvi_baseline_2000_2004',
    'kitui_rainfall_baseline_1981_2010',
    'kitui_rainfall_recent_2020_2024',
    'kitui_short_rains_mean',
    'kitui_long_rains_mean',
    'kitui_elevation_srtm30',
    'kitui_slope_deg',
    'kitui_aspect_deg',
    'kitui_worldpop_2020',
    'kitui_jrc_water_seasonality',
    'kitui_jrc_water_occurrence',
    'kitui_jrc_water_transition',
    'kitui_lst_mean_celsius',
    'kitui_lst_dry_season_celsius',
    'kitui_et_mean_mm_8day',
    'kitui_soil_moisture_mean',
    'kitui_soil_moisture_dry',
    'kitui_soil_moisture_wet',
]

print('── Asset verification ────────────────────────────────────')
missing  = []
existing = []
for name in EXPECTED_ASSETS:
    asset_id = f'{ASSET_FOLDER}/{name}'
    try:
        ee.data.getAsset(asset_id)
        print(f'  ✅ {name}')
        existing.append(name)
    except Exception:
        print(f'  ❌ {name}  ← MISSING')
        missing.append(name)

print()
if missing:
    print(f'⚠  {len(missing)} asset(s) missing — exports still running or not yet submitted.')
    print(f'   Check: https://code.earthengine.google.com/tasks')
    if not existing:
        print('   No assets ready yet — re-run this cell after exports complete.')
else:
    print(f'✅ All {len(EXPECTED_ASSETS)} assets verified. Ready for Notebook 02.')

# ── QA map — only loads layers that actually exist ─────────────────────────
if not existing:
    print('\nQA map skipped — no assets available yet.')
else:
    print(f'\n── QA map ({len(existing)} of {len(EXPECTED_ASSETS)} layers) ──────────────────────────')
    Map2 = geemap.Map()
    Map2.centerObject(kitui_geom, 8)
    Map2.addLayer(kitui_geom, {'color': '0B5394'}, 'Kitui boundary')

    # Only add layers whose assets exist
    VIS_LAYERS = [
        ('kitui_ndvi_mean_2000_2025',
         {'min': 0.0, 'max': 0.8, 'palette': ['#d7191c','#ffffbf','#1a9641']},
         'NDVI mean 2000–2025'),
        ('kitui_rainfall_baseline_1981_2010',
         {'min': 200, 'max': 1200, 'palette': ['#d7191c','#ffffbf','#2c7bb6']},
         'Rainfall baseline 1981–2010 (mm/yr)'),
        ('kitui_slope_deg',
         {'min': 0, 'max': 30, 'palette': ['white','#888888','black']},
         'Slope (degrees)'),
        ('kitui_jrc_water_seasonality',
         {'min': 0, 'max': 12, 'palette': ['white','#9DC3E6','#2E75B6']},
         'Water seasonality (JRC)'),
        ('kitui_soil_moisture_mean',
         {'min': 0.05, 'max': 0.40, 'palette': ['#d7191c','#ffffbf','#2c7bb6']},
         'Soil moisture mean (ERA5-Land)'),
    ]

    for asset_name, vis, label in VIS_LAYERS:
        if asset_name in existing:
            Map2.addLayer(
                ee.Image(f'{ASSET_FOLDER}/{asset_name}'),
                vis, label
            )

    Map2.add_layer_control()
    print('Toggle layers to verify coverage and values')
    display(Map2)


── Asset verification ────────────────────────────────────
  ✅ kitui_ndvi_mean_2000_2025
  ✅ kitui_ndvi_baseline_2000_2004
  ✅ kitui_rainfall_baseline_1981_2010
  ✅ kitui_rainfall_recent_2020_2024
  ✅ kitui_short_rains_mean
  ✅ kitui_long_rains_mean
  ✅ kitui_elevation_srtm30
  ✅ kitui_slope_deg
  ✅ kitui_aspect_deg
  ✅ kitui_worldpop_2020
  ✅ kitui_jrc_water_seasonality
  ✅ kitui_jrc_water_occurrence
  ✅ kitui_jrc_water_transition
  ✅ kitui_lst_mean_celsius
  ✅ kitui_lst_dry_season_celsius
  ✅ kitui_et_mean_mm_8day
  ✅ kitui_soil_moisture_mean
  ✅ kitui_soil_moisture_dry
  ✅ kitui_soil_moisture_wet

✅ All 19 assets verified. Ready for Notebook 02.

── QA map (19 of 19 layers) ──────────────────────────
Toggle layers to verify coverage and values


Map(center=[-1.869528123546361, 38.446305795184365], controls=(WidgetControl(options=['position', 'transparent…